# Generalised Zero-Shot — Multi-Model Clinical Summarisation
Zero-shot summarisation with a fixed 150-token budget, generalised across the same 4 models as the ISP notebook.

**Switch models by changing only `SELECTED_MODEL` in Cell 2.** Supported: `qwen2.5`, `ministral_8b`, `llama3.1_8b`, `qwen3`.

Matches the ISP notebook: fixed 150 tokens, no word-count instruction, BERTScore = roberta-large (no baseline), summary grammar per case, prompt grammar per run, summarisation score = 0.6·ROUGE-L + 0.4·BERT-F1. CSV columns match the ISP final CSV (grammar_score + summarisation_score added; token-count/above_minimum columns dropped).

In [ ]:
# ========================= Cell 1 — Install dependencies =========================
import subprocess
subprocess.run(["pip","install","-q","transformers","accelerate","bitsandbytes",
    "datasets","evaluate","rouge-score","bert-score","pandas","tqdm","matplotlib",
    "language_tool_python==2.9.2","nltk","sentencepiece"], check=True)
subprocess.run(["pip","install","-q","-U","transformers"], check=True)
print("Dependencies installed")

In [ ]:
# ========================= Cell 2 — Imports, model registry & config =========================
import os, re, gc, json, torch
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
import evaluate as hf_evaluate
from bert_score import score as bert_score_fn
import nltk, language_tool_python
nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True)

# ╔══════════════════════════════════════════════════════════════╗
# ║  CHANGE ONLY THIS LINE TO SWITCH MODELS                       ║
# ╚══════════════════════════════════════════════════════════════╝
SELECTED_MODEL = "qwen2.5"   # qwen2.5 | ministral_8b | llama3.1_8b | qwen3

MODEL_REGISTRY = {
    "qwen2.5":      {"repo": "Qwen/Qwen2.5-7B-Instruct",            "gated": False, "dtype": torch.float16,  "display": "Qwen2.5-7B"},
    "ministral_8b": {"repo": "mistralai/Ministral-8B-Instruct-2410", "gated": True,  "dtype": torch.bfloat16, "display": "Ministral-8B"},
    "llama3.1_8b":  {"repo": "meta-llama/Llama-3.1-8B-Instruct",     "gated": True,  "dtype": torch.bfloat16, "display": "Llama-3.1-8B"},
    "qwen3":        {"repo": "Qwen/Qwen3-8B",                        "gated": False, "dtype": torch.bfloat16, "display": "Qwen3-8B"},
}
assert SELECTED_MODEL in MODEL_REGISTRY
_cfg          = MODEL_REGISTRY[SELECTED_MODEL]
GEN_MODEL     = _cfg["repo"]
MODEL_GATED   = _cfg["gated"]
MODEL_TAG     = SELECTED_MODEL
MODEL_DISPLAY = _cfg["display"]

NLI_MODEL   = "cross-encoder/nli-deberta-v3-large"
BERT_MODEL  = "roberta-large"          # match ISP (no baseline rescaling)
FIXED_TOKENS = 150                     # match ISP: max_new = min_new = 150
W_ROUGEL, W_BERT = 0.6, 0.4

SEED = 42
torch.manual_seed(SEED)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
DEVICE_NLI = 0 if torch.cuda.is_available() else -1

# ── Compute dtype: fp16 is fast on all GPUs (bf16 slow on T4). ──
FORCE_FP16 = True
if FORCE_FP16:
    MODEL_DTYPE = torch.float16
elif torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    MODEL_DTYPE = _cfg["dtype"]
else:
    MODEL_DTYPE = torch.float16

# ── Paths (model-tagged so runs don't overwrite each other) ──
WORK_DIR     = "/kaggle/working"
DATA_ROOT    = "/kaggle/input/datasets/anshumanmoharana07/biolaysumm/Data/Short"
FULLTEXT_DIR = os.path.join(DATA_ROOT, "fulltext")
SUMMARY_DIR  = os.path.join(DATA_ROOT, "summaries")
os.makedirs(WORK_DIR, exist_ok=True)
def _p(n): return os.path.join(WORK_DIR, f"{MODEL_TAG}_{n}")
OUTPUT_CSV      = _p("zeroshot.csv")
CHECKPOINT_JSON = _p("zeroshot_checkpoint.json")
METRICS_JSON    = _p("zeroshot_metrics.json")
TOP10_ROUGEL_PNG = _p("top10_high_rougeL.png")
BOT10_ROUGEL_PNG = _p("top10_low_rougeL.png")
TOP10_BERTF1_PNG = _p("top10_high_bert_f1.png")
BOT10_BERTF1_PNG = _p("top10_low_bert_f1.png")

print("Model:", GEN_MODEL, "| tag:", MODEL_TAG, "| gated:", MODEL_GATED)
print("dtype:", MODEL_DTYPE, "| FIXED_TOKENS:", FIXED_TOKENS, "| BERT:", BERT_MODEL)
print("CUDA:", torch.cuda.is_available())

In [ ]:
# ========================= Cell 3 — HuggingFace token (gated models) =========================
HF_TOKEN_FALLBACK = ""   # optionally paste hf_xxx here
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN"); print("HF token from Kaggle Secrets")
except Exception: pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if HF_TOKEN: print("HF token from environment")
if not HF_TOKEN and HF_TOKEN_FALLBACK:
    HF_TOKEN = HF_TOKEN_FALLBACK; print("HF token from fallback")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN; os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login; login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception as e: print("hub login skipped:", e)
else:
    print("No HF token found (open models work; gated Llama/Ministral will fail).")
if MODEL_GATED and not HF_TOKEN:
    raise RuntimeError(f"'{SELECTED_MODEL}' is gated — set HF_TOKEN and request access to {GEN_MODEL}.")

In [ ]:
# ========================= Cell 4 — Load paired dataset =========================
def load_pairs(full_dir, sum_dir):
    rows, files = [], sorted([f for f in os.listdir(full_dir) if f.endswith(".txt")])
    for f in files:
        fn = f[:-4]; sp = os.path.join(sum_dir, f"{fn}_sum.txt")
        if not os.path.exists(sp): continue
        with open(os.path.join(full_dir, f), "r", encoding="utf-8", errors="ignore") as a: case = a.read().strip()
        with open(sp, "r", encoding="utf-8", errors="ignore") as b: ref = b.read().strip()
        if case and ref: rows.append({"filename": fn, "clinical_case": case, "reference_summary": ref})
    return pd.DataFrame(rows)

df_data = load_pairs(FULLTEXT_DIR, SUMMARY_DIR)
print("Loaded pairs:", len(df_data))
df_data.head(2)

In [ ]:
# ========================= Cell 5 — Optional reset =========================
RESET_PREVIOUS = False   # True for a clean re-run of THIS model
if RESET_PREVIOUS:
    for p in [OUTPUT_CSV, CHECKPOINT_JSON, METRICS_JSON,
              TOP10_ROUGEL_PNG, BOT10_ROUGEL_PNG, TOP10_BERTF1_PNG, BOT10_BERTF1_PNG]:
        if os.path.exists(p): os.remove(p); print("Deleted:", p)
else:
    print("Keeping previous outputs/checkpoints for", MODEL_TAG, "if they exist.")

In [ ]:
# ========================= Cell 6 — Load model, metrics, NLI =========================
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=MODEL_DTYPE, bnb_4bit_use_double_quant=True,
)
_auth = {"token": HF_TOKEN} if HF_TOKEN else {}

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL, trust_remote_code=True, **_auth)
if gen_tokenizer.pad_token is None:
    gen_tokenizer.pad_token = gen_tokenizer.eos_token
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL, quantization_config=bnb, device_map="auto",
    trust_remote_code=True, low_cpu_mem_usage=True, **_auth)
gen_model.eval(); gen_model.config.use_cache = True
print("Generation model loaded on", next(gen_model.parameters()).device)

def build_chat_text(messages):
    """Apply chat template; disable Qwen3 thinking mode where supported."""
    try:
        return gen_tokenizer.apply_chat_template(messages, tokenize=False,
                                                 add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

rouge_metric = hf_evaluate.load("rouge")
nli_pipe = pipeline("text-classification", model=NLI_MODEL, tokenizer=NLI_MODEL,
                    top_k=None, device=DEVICE_NLI, truncation=True, max_length=512)
language_tool = language_tool_python.LanguageTool("en-US")
print("Metrics + NLI + LanguageTool ready")

In [ ]:
# ========================= Cell 7 — Utility functions =========================
SCHEMA_COLS = [
    "filename", "reference_summary", "generated_summary",
    "entailment_prob_reference", "neutral_prob_reference", "contradiction_prob_reference", "reference_entailed",
    "entailment_prob_generated",  "neutral_prob_generated",  "contradiction_prob_generated",  "generated_entailed",
    "both_entailed",
    "rouge1", "rouge2", "rougeL",
    "bert_precision", "bert_recall", "bert_f1",
    "grammar_score", "summarisation_score",
    "within_budget", "is_complete_sentence", "method",
]

def tok_len(text):
    return len(gen_tokenizer.encode(text, add_special_tokens=False))
def ends_complete(text):
    return bool(re.search(r"[.!?]\s*$", text.strip()))
def remove_incomplete_sentence(text):
    t = text.strip(); stops = list(re.finditer(r"[.!?]", t))
    return t[:stops[-1].end()].strip() if stops else ""
def trunc_to_tokens(text, max_tokens):
    ids = gen_tokenizer.encode(text, add_special_tokens=False)
    if len(ids) <= max_tokens: return text
    return gen_tokenizer.decode(ids[:max_tokens], skip_special_tokens=True).strip()

# ── Zero-shot prompt: NO word-count instruction (matches ISP) ──
def build_prompt_body():
    return (
        "Write a concise clinical summary of the following case report in clear, continuous prose.\n"
        "Your summary MUST cover ALL of the following:\n"
        "1. Patient demographics (age and sex)\n"
        "2. Presenting complaint and relevant medical history\n"
        "3. Key clinical findings and investigations\n"
        "4. Diagnosis\n"
        "5. Treatment plan\n"
        "6. Clinical outcome\n"
        "Use clear, professional medical language.\n"
        "Do NOT use bullet points or headers.\n"
        "Always end each sentence with proper punctuation.\n"
        "Complete every sentence fully before stopping."
    )
ZERO_SHOT_PROMPT = build_prompt_body()   # fixed for the whole run

def finalize_summary(raw_text):
    t = trunc_to_tokens(raw_text.strip(), FIXED_TOKENS)
    if not ends_complete(t): t = remove_incomplete_sentence(t)
    if not t: t = "Summary unavailable."
    t = trunc_to_tokens(t, FIXED_TOKENS)
    if not ends_complete(t): t = remove_incomplete_sentence(t) or "Summary unavailable."
    return t, tok_len(t), (tok_len(t) <= FIXED_TOKENS), ends_complete(t)

def generate_summary(case_text):
    """Single-pass, fixed 150 tokens, chat-template formatted."""
    for in_lim in [2048, 1536, 1024, 768, 512, 384, 256]:
        case_ids   = gen_tokenizer(case_text, add_special_tokens=False, truncation=True, max_length=in_lim)["input_ids"]
        short_case = gen_tokenizer.decode(case_ids, skip_special_tokens=True)
        messages = [
            {"role": "system", "content": "You are a clinical summarization expert."},
            {"role": "user",   "content": f"{ZERO_SHOT_PROMPT}\n\nClinical Case Report:\n{short_case}"},
        ]
        text = build_chat_text(messages)
        try:
            inputs = gen_tokenizer(text, return_tensors="pt").to(gen_model.device)
            with torch.inference_mode():
                out = gen_model.generate(
                    **inputs, max_new_tokens=FIXED_TOKENS, min_new_tokens=FIXED_TOKENS,
                    do_sample=False, repetition_penalty=1.2,
                    eos_token_id=gen_tokenizer.eos_token_id, pad_token_id=gen_tokenizer.pad_token_id)
            raw = gen_tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
            ft, g_tok, wb, ic = finalize_summary(raw)
            return {"generated_summary": ft, "within_budget": wb, "is_complete_sentence": ic}
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
    return {"generated_summary": "Summary unavailable.", "within_budget": True, "is_complete_sentence": False}

# ── Grammar scoring (same loaded model + LanguageTool fallback) ──
def complete_grammar(messages, max_tokens=5):
    text = build_chat_text(messages)
    mi = gen_tokenizer([text], return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        gen_ids = gen_model.generate(**mi, max_new_tokens=max_tokens, temperature=0.1,
                                     do_sample=False, pad_token_id=gen_tokenizer.pad_token_id,
                                     eos_token_id=gen_tokenizer.eos_token_id)
    out_ids = gen_ids[0][len(mi.input_ids[0]):].tolist()
    return gen_tokenizer.decode(out_ids, skip_special_tokens=True).strip("\n").strip()

def language_tool_fallback(text):
    matches = language_tool.check(text); score = 100
    for m in matches:
        if m.ruleId.startswith("MORFOLOGIK_") or m.ruleId == "UPPERCASE_SENTENCE_START": score -= 5
        elif "AGREEMENT" in m.ruleId: score -= 15
        elif "GRAMMAR" in m.ruleId or "SENTENCE" in m.ruleId: score -= 20
        else: score -= 10
    return max(0, score)

def get_llm_grammar_score(text):
    if not text or not text.strip(): return 0
    messages = [
        {"role": "system", "content": "You are a strict grammar evaluator. Score grammar from 0 to 100. 100 = perfect grammar. 0 = very poor grammar. Return only a number, no text."},
        {"role": "user",   "content": 'Evaluate the grammar of this text and return a score from 0 to 100.\nText:\n"""\n' + text + '\n"""'},
    ]
    try:
        raw = complete_grammar(messages, max_tokens=5)
        mt = re.match(r"^\[?(\d+)\]?$", raw.strip())
        if mt: score = int(mt.group(1))
        else:
            nums = re.findall(r"\d+", raw); score = int(nums[0]) if nums else None
            if score is None: raise ValueError("no number")
        return score if 0 <= score <= 100 else language_tool_fallback(text)
    except (ValueError, TypeError):
        return language_tool_fallback(text)
    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            torch.cuda.empty_cache(); gc.collect(); return language_tool_fallback(text)
        raise

# ── ROUGE / BERT (BERT matches ISP: roberta-large, no baseline) ──
def rouge_scores(pred, ref):
    r = rouge_metric.compute(predictions=[pred], references=[ref], use_stemmer=True)
    return float(r["rouge1"]), float(r["rouge2"]), float(r["rougeL"])
def bert_scores(pred, ref):
    P, R, F1 = bert_score_fn([pred], [ref], model_type=BERT_MODEL, lang="en", verbose=False)
    return float(P[0]), float(R[0]), float(F1[0])

# ── NLI ──
LABEL_MAP = {"LABEL_0":"CONTRADICTION","LABEL_1":"NEUTRAL","LABEL_2":"ENTAILMENT",
             "ENTAILMENT":"ENTAILMENT","NEUTRAL":"NEUTRAL","CONTRADICTION":"CONTRADICTION"}
def unwrap_nli_output(raw):
    if isinstance(raw, dict): return [raw]
    if isinstance(raw, list) and len(raw) > 0: return raw[0] if isinstance(raw[0], list) else raw
    raise ValueError(f"Unrecognised NLI output: {raw!r}")
def parse_nli(raw_output):
    items = unwrap_nli_output(raw_output)
    if not items: raise ValueError("Empty NLI output.")
    probs = {"ENTAILMENT":0.0,"NEUTRAL":0.0,"CONTRADICTION":0.0}
    for item in items:
        canonical = LABEL_MAP.get(str(item["label"]).upper().strip())
        if canonical: probs[canonical] = float(item["score"])
    ent, neu, con = probs["ENTAILMENT"], probs["NEUTRAL"], probs["CONTRADICTION"]
    return ent, neu, con, (ent > neu) and (ent > con)
def nli_entailment(premise, hypothesis):
    premise, hypothesis = str(premise)[:3600], str(hypothesis)[:1500]
    try:
        return parse_nli(nli_pipe([[premise, hypothesis]]))
    except Exception as e1:
        try:
            return parse_nli(nli_pipe({"text": premise, "text_pair": hypothesis}))
        except Exception as e2:
            print(f"[NLI WARN] {e1} | {e2}"); return float("nan"), float("nan"), float("nan"), False

# ── Checkpointing ──
def save_checkpoint(rows, done_files):
    d = pd.DataFrame(rows) if rows else pd.DataFrame(columns=SCHEMA_COLS)
    for c in SCHEMA_COLS:
        if c not in d.columns: d[c] = None
    tmp = OUTPUT_CSV + ".tmp"; d[SCHEMA_COLS].to_csv(tmp, index=False); os.replace(tmp, OUTPUT_CSV)
    tmp2 = CHECKPOINT_JSON + ".tmp"
    with open(tmp2, "w") as f:
        json.dump({"processed_count": len(done_files), "processed_filenames": sorted(list(done_files))}, f, indent=2)
    os.replace(tmp2, CHECKPOINT_JSON)
def load_checkpoint():
    rows, done = [], set()
    if os.path.exists(OUTPUT_CSV) and os.path.getsize(OUTPUT_CSV) > 0:
        try:
            old = pd.read_csv(OUTPUT_CSV)
            if not old.empty:
                rows = old.to_dict("records")
                done = set(old["filename"].astype(str).tolist()) if "filename" in old.columns else set()
        except Exception as e: print("[WARN] CSV read failed:", e)
    if os.path.exists(CHECKPOINT_JSON) and os.path.getsize(CHECKPOINT_JSON) > 0:
        try:
            with open(CHECKPOINT_JSON) as f: ck = json.load(f)
            done |= set(ck.get("processed_filenames", []))
        except Exception as e: print("[WARN] ckpt read failed:", e)
    return rows, done

print("Utilities ready (fixed-150, chat-template, grammar, NLI, ROUGE/BERT matched to ISP)")

In [ ]:
# ========================= Cell 8 — Prompt grammar score (once per run) =========================
PROMPT_GRAMMAR = get_llm_grammar_score(ZERO_SHOT_PROMPT)
print(f"Zero-shot prompt grammar score: {PROMPT_GRAMMAR} / 100")
print("Prompt:\n" + ZERO_SHOT_PROMPT)

In [ ]:
# ========================= Cell 9 — Main run (checkpoint every 10) =========================
rows, done = load_checkpoint()
print("[RESUME] already done:", len(done))
todo = df_data[~df_data["filename"].isin(done)].reset_index(drop=True)
print("[RESUME] remaining:", len(todo), "/", len(df_data))

for i in tqdm(range(len(todo)), desc=f"ZeroShot {MODEL_TAG}"):
    ex = todo.iloc[i]; fn = ex["filename"]; case = ex["clinical_case"]; ref = ex["reference_summary"]

    try:
        g = generate_summary(case); pred = g["generated_summary"]
    except Exception as e:
        print(f"[ERROR gen] {fn}: {type(e).__name__}: {e}"); torch.cuda.empty_cache(); gc.collect(); continue

    try:
        r1, r2, rL  = rouge_scores(pred, ref)
        bp, br, bf1 = bert_scores(pred, ref)
    except Exception as e:
        print(f"[ERROR metrics] {fn}: {e}"); r1=r2=rL=bp=br=bf1=float("nan")

    grammar = get_llm_grammar_score(pred)
    summ = round(W_ROUGEL*rL + W_BERT*bf1, 4) if (rL==rL and bf1==bf1) else float("nan")

    ent_ref, neu_ref, con_ref, ref_ent = nli_entailment(case, ref)
    ent_gen, neu_gen, con_gen, gen_ent = nli_entailment(case, pred)
    both_ent = bool(ref_ent) and bool(gen_ent)

    rows.append({
        "filename": fn, "reference_summary": ref, "generated_summary": pred,
        "entailment_prob_reference": ent_ref, "neutral_prob_reference": neu_ref,
        "contradiction_prob_reference": con_ref, "reference_entailed": ref_ent,
        "entailment_prob_generated": ent_gen, "neutral_prob_generated": neu_gen,
        "contradiction_prob_generated": con_gen, "generated_entailed": gen_ent,
        "both_entailed": both_ent,
        "rouge1": r1, "rouge2": r2, "rougeL": rL,
        "bert_precision": bp, "bert_recall": br, "bert_f1": bf1,
        "grammar_score": grammar, "summarisation_score": summ,
        "within_budget": g["within_budget"], "is_complete_sentence": g["is_complete_sentence"],
        "method": f"zeroshot_{MODEL_TAG}_fixed150",
    })
    done.add(fn)
    if (len(done) % 10) == 0:
        save_checkpoint(rows, done); print(f"[CHECKPOINT] {len(done)} files")
    torch.cuda.empty_cache(); gc.collect()

save_checkpoint(rows, done)
print("\n[FINAL SAVE]", OUTPUT_CSV, "| rows:", len(rows))

In [ ]:
# ========================= Cell 10 — Aggregate metrics =========================
d = pd.read_csv(OUTPUT_CSV)
avg_rougeL  = round(pd.to_numeric(d["rougeL"], errors="coerce").mean(), 4)
avg_bert_f1 = round(pd.to_numeric(d["bert_f1"], errors="coerce").mean(), 4)
avg_grammar = round(pd.to_numeric(d["grammar_score"], errors="coerce").mean(), 2)
summ_score  = round(W_ROUGEL*avg_rougeL + W_BERT*avg_bert_f1, 4)

metrics = {
    "model": MODEL_TAG, "model_repo": GEN_MODEL, "n_cases": int(len(d)),
    "fixed_tokens": FIXED_TOKENS, "method": "zeroshot",
    "avg_rougeL": avg_rougeL, "avg_bert_f1": avg_bert_f1,
    "avg_summary_grammar_score": avg_grammar,
    "prompt_grammar_score": PROMPT_GRAMMAR,
    "summarisation_score": summ_score,
}
with open(METRICS_JSON, "w") as f: json.dump(metrics, f, indent=2)

print("="*60)
print(f"  ZERO-SHOT AGGREGATE METRICS — {MODEL_DISPLAY} ({len(d)} cases)")
print("="*60)
print(f"  Average ROUGE-L         : {avg_rougeL}")
print(f"  Average BERTScore (F1)  : {avg_bert_f1}")
print(f"  Avg summary grammar     : {avg_grammar} / 100")
print(f"  Prompt grammar          : {PROMPT_GRAMMAR} / 100")
print(f"  Summarisation score     : {summ_score}  (= {W_ROUGEL}*ROUGE-L + {W_BERT}*BERT-F1)")
print("="*60)
print(f"  Both entailed           : {pd.to_numeric(d['both_entailed'].astype(float), errors='coerce').mean()*100:.1f}%")
print(f"  Complete sentences      : {pd.to_numeric(d['is_complete_sentence'].astype(float), errors='coerce').mean()*100:.1f}%")
print("  Saved metrics ->", METRICS_JSON)

In [ ]:
# ========================= Cell 11 — Top/Bottom 10 plots =========================
FALLBACK_TEXT = "Summary unavailable."
def plot_top_bottom(df, score_col, top_n, high_color, low_color, high_title, low_title, high_path, low_path):
    x = df.copy(); x[score_col] = pd.to_numeric(x[score_col], errors="coerce")
    x = x.dropna(subset=[score_col])
    if "generated_summary" in x.columns:
        x = x[x["generated_summary"].astype(str).str.strip() != FALLBACK_TEXT]
    if x.empty: print(f"[WARN] no rows for {score_col}"); return
    for subset, color, title, path, asc in [
        (x.sort_values(score_col, ascending=False).drop_duplicates("filename").head(top_n), high_color, high_title, high_path, False),
        (x.sort_values(score_col, ascending=True ).drop_duplicates("filename").head(top_n), low_color,  low_title,  low_path,  True),
    ]:
        s = subset.sort_values(score_col, ascending=asc)
        names = [f.replace("multiclinsum_gs_en_", "case_") for f in s["filename"]]
        vals = s[score_col].tolist()
        fig, ax = plt.subplots(figsize=(13,6))
        bars = ax.bar(range(len(names)), vals, color=color, edgecolor="black", linewidth=0.7, alpha=0.85)
        for b,v in zip(bars, vals): ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.002, f"{v:.4f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
        ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha="right", fontsize=9)
        lo, hi = min(vals), max(vals); pad = max(0.01, (hi-lo)*0.15)
        ax.set_ylim(max(0, lo-pad), min(1.0, hi+pad)); ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
        ax.set_title(f"{title}\n({MODEL_DISPLAY} zero-shot)", fontsize=13, fontweight="bold", pad=15)
        ax.grid(axis="y", linestyle="--", alpha=0.5); ax.set_axisbelow(True)
        plt.tight_layout(); plt.savefig(path, dpi=150, bbox_inches="tight"); plt.show(); print("[SAVED]", path)

d = pd.read_csv(OUTPUT_CSV)
plot_top_bottom(d, "rougeL", 10, "#2ecc71", "#e74c3c", "Top 10 — Highest ROUGE-L", "Top 10 — Lowest ROUGE-L", TOP10_ROUGEL_PNG, BOT10_ROUGEL_PNG)
plot_top_bottom(d, "bert_f1", 10, "#3498db", "#e67e22", "Top 10 — Highest BERTScore F1", "Top 10 — Lowest BERTScore F1", TOP10_BERTF1_PNG, BOT10_BERTF1_PNG)
print("\nDone —", MODEL_TAG)